# DGD-TabPA — Full Experiment Workflow

This notebook documents the **complete sequence** to produce evaluation evidence:

1. Environment setup & data download  
2. Single end-to-end run (train → distill → evaluate → figures)  
3. Multi-dataset run on **all 10 benchmarks**  
4. SMOTE baseline comparison  
5. Ablation studies  
6. Privacy–utility sweep  
7. Collect results for tables & figures  

**Outputs land in** outputs/experiments/.

> Tip: for a quick demo, keep EPOCHS / DISTILL_EPOCHS low. For stronger numbers, use the recommended settings in the README.


## 0. Configuration

Adjust these knobs once; later cells reuse them.

In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD if (CWD / "scripts" / "run_experiment.py").exists() else CWD.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

CONFIG = "config/default.yaml"
PRIMARY_DATASET = "diabetes"
EPOCHS = 30
DISTILL_EPOCHS = 50
BATCH_SUITE_EPOCHS = 30
TRAIN_ALL_TEN = True  # train/eval on all 10 benchmarks
RUN_HEAVY = False     # also ablations + privacy + smote suite

EXPERIMENTS_DIR = PROJECT_ROOT / "outputs" / "experiments"
MASTER_CSV = EXPERIMENTS_DIR / "results_master.csv"

print(f"Project root : {PROJECT_ROOT}")
print(f"Experiments  : {EXPERIMENTS_DIR}")
print(f"TRAIN_ALL_TEN: {TRAIN_ALL_TEN}")
print(f"RUN_HEAVY    : {RUN_HEAVY}")


In [ ]:
def run_cmd(args, check=True):
    """Run a project script and stream output."""
    cmd = [sys.executable, *args]
    print("\n>>>", " ".join(cmd))
    result = subprocess.run(cmd, cwd=str(PROJECT_ROOT))
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result.returncode

def show_figures(run_id, max_figs=8):
    fig_dir = EXPERIMENTS_DIR / run_id / "figures"
    if not fig_dir.exists():
        print(f"No figures at {fig_dir}")
        return
    paths = sorted(fig_dir.glob("*.png"))[:max_figs]
    for p in paths:
        display(Markdown(f"**{p.name}**"))
        display(Image(filename=str(p)))

def load_summary(run_id):
    path = EXPERIMENTS_DIR / run_id / "summary_row.json"
    if not path.exists():
        print(f"Missing {path}")
        return None
    with open(path, encoding="utf-8") as f:
        return json.load(f)

---
## Step 1 — Install dependencies

Run once per environment.

In [ ]:
# Uncomment if packages are missing:
# run_cmd(["-m", "pip", "install", "-r", "requirements.txt"])
print("Assuming requirements are installed. See README Setup section.")

---
## Step 2 — Download benchmark datasets

Writes CSVs into `data/raw/`.

In [ ]:
# Full suite (all datasets in config):
run_cmd(["scripts/download_data.py"])

# Or a single dataset:
# run_cmd(["scripts/download_data.py", "--dataset", PRIMARY_DATASET])

list((PROJECT_ROOT / "data" / "raw").glob("*.csv"))

---
## Step 3 — Single end-to-end DGD-TabPA run

**Pipeline inside `run_experiment.py`:**

1. Load & preprocess data  
2. Train diffusion model  
3. Distill synthetic subset  
4. Evaluate fidelity / TSTR / privacy (DCR + MIA)  
5. Save `metrics.json`, figures, append `results_master.csv`

This is the minimum complete process for one dataset.

In [ ]:
RUN_ID = f"{PRIMARY_DATASET}_dgd"

run_cmd([
    "scripts/run_experiment.py",
    "--config", CONFIG,
    "--dataset", PRIMARY_DATASET,
    "--method", "dgd_tabpa",
    "--epochs", str(EPOCHS),
    "--distill-epochs", str(DISTILL_EPOCHS),
    "--run-id", RUN_ID,
])

summary = load_summary(RUN_ID)
display(summary)
show_figures(RUN_ID)

---
## Step 4 — SMOTE baseline (same evaluation protocol)

Needed for utility vs privacy comparison against DGD-TabPA.

In [ ]:
SMOTE_ID = f"{PRIMARY_DATASET}_smote"

run_cmd([
    "scripts/run_experiment.py",
    "--config", CONFIG,
    "--dataset", PRIMARY_DATASET,
    "--method", "smote",
    "--run-id", SMOTE_ID,
])

display(pd.DataFrame([load_summary(RUN_ID), load_summary(SMOTE_ID)]))

---
## Step 5 — Multi-dataset training (all 10)

ll_ten = adult, churn, credit, covertype, cvd, hcv, ilpd, diabetes, california_housing, king_county

Set TRAIN_ALL_TEN = True in Step 0 to execute.


In [ ]:
if TRAIN_ALL_TEN:
    run_cmd([
        "scripts/run_batch_experiments.py",
        "--suite", "all_ten",
        "--epochs", str(BATCH_SUITE_EPOCHS),
        "--distill-epochs", str(DISTILL_EPOCHS),
    ])
else:
    print("Skipped (set TRAIN_ALL_TEN=True). Equivalent CLI:")
    print(f"  python scripts/run_batch_experiments.py --suite all_ten --epochs {BATCH_SUITE_EPOCHS}")


---
## Step 6 — Ablation studies

| Flag | What it tests |
|---|---|
| `mlp_denoiser` | Transformer vs MLP denoiser |
| `no_attention` | Conditioning Attention on/off |
| `minmax` | Quantile vs min-max preprocessing |
| `raw_space` | Latent vs raw-space distillation |

In [ ]:
if RUN_HEAVY:
    run_cmd([
        "scripts/run_batch_experiments.py",
        "--suite", "ablations",
        "--dataset", PRIMARY_DATASET,
        "--epochs", str(BATCH_SUITE_EPOCHS),
    ])
else:
    print("Skipped. Example single ablation:")
    print(
        f"  python scripts/run_experiment.py --dataset {PRIMARY_DATASET} "
        f"--ablation mlp_denoiser --epochs {EPOCHS} --distill-epochs {DISTILL_EPOCHS}"
    )

---
## Step 7 — Privacy–utility sweep

Runs non-private + ε ∈ {1, 4, 8, 100} and builds a trade-off plot when possible.

In [ ]:
if RUN_HEAVY:
    run_cmd([
        "scripts/run_batch_experiments.py",
        "--suite", "privacy",
        "--dataset", PRIMARY_DATASET,
        "--epochs", str(BATCH_SUITE_EPOCHS),
    ])
    tradeoff = EXPERIMENTS_DIR / f"privacy_utility_{PRIMARY_DATASET}.png"
    if tradeoff.exists():
        display(Image(filename=str(tradeoff)))
else:
    print("Skipped. Equivalent CLI:")
    print(
        f"  python scripts/run_batch_experiments.py --suite privacy "
        f"--dataset {PRIMARY_DATASET} --epochs {BATCH_SUITE_EPOCHS}"
    )

---
## Step 8 — Multi-dataset SMOTE baselines

Optional but recommended for a fair comparative table.

In [ ]:
if RUN_HEAVY:
    run_cmd([
        "scripts/run_batch_experiments.py",
        "--suite", "smote",
        "--epochs", "1",
    ])
else:
    print("Skipped. Equivalent CLI:")
    print("  python scripts/run_batch_experiments.py --suite smote")

---
## Step 9 — Collect results

Use `results_master.csv` for reporting tables. Figures live under each `run_id/figures/`.

In [ ]:
if MASTER_CSV.exists():
    master = pd.read_csv(MASTER_CSV)
    print(f"Rows in master table: {len(master)}")
    display(master)

    # Compact reporting view
    cols = [
        c for c in [
            "dataset", "method",
            "wasserstein_mean", "pcd", "jsd_mean",
            "mean_tstr_f1", "mean_f1_gap", "mean_tstr_auc",
            "dcr_median", "dcr_5th_percentile", "mia_auc",
            "privacy_rating", "epsilon",
        ] if c in master.columns
    ]
    display(master[cols].round(4))

    # Save a cleaned table for Word/Excel paste
    out_table = EXPERIMENTS_DIR / "results_table.csv"
    master[cols].to_csv(out_table, index=False)
    print(f"Saved results table -> {out_table}")
else:
    print(f"No master CSV yet at {MASTER_CSV}. Run Steps 3+ first.")

In [ ]:
# Inventory of all produced figure PNGs
fig_files = sorted(EXPERIMENTS_DIR.glob("**/figures/*.png"))
print(f"Total figures: {len(fig_files)}")
for p in fig_files[:40]:
    print(" -", p.relative_to(PROJECT_ROOT))
if len(fig_files) > 40:
    print(f" ... and {len(fig_files) - 40} more")

---
## Optional — Training-only / inspect checkpoints

Useful for debugging, not required if you use `run_experiment.py`.

In [ ]:
# Training only (no distill/eval):
# run_cmd(["scripts/train.py", "--config", CONFIG, "--dataset", PRIMARY_DATASET, "--epochs", str(EPOCHS)])

# Inspect an existing checkpoint under outputs/:
# run_cmd(["scripts/inspect_outputs.py", "--dataset", PRIMARY_DATASET, "--n-samples", "200"])

print("Optional helpers documented in README.")

---
## Checklist — Fully completed process

| # | Step | Script | Purpose |
|---|---|---|---|
| 1 | Install | `pip install -r requirements.txt` | Environment |
| 2 | Download data | `scripts/download_data.py` | Benchmarks |
| 3 | DGD single/full | `scripts/run_experiment.py` / `--suite core_fast` | Quantitative metrics |
| 4 | SMOTE baseline | `--method smote` / `--suite smote` | Baseline comparison |
| 5 | Ablations | `--suite ablations` | Component ablations |
| 6 | Privacy sweep | `--suite privacy` | Privacy–utility trade-off |
| 7 | Collect tables/figures | `results_master.csv` + `figures/` | Reporting from evidence |

**Done when:** `results_master.csv` has DGD + SMOTE (+ ablation/privacy) rows, and each run folder contains `metrics.json` plus figure PNGs ready for reporting.